In [2]:
# ========================== CONFIG ==========================
import os
import re
from datetime import date, timedelta, datetime
from zoneinfo import ZoneInfo

# ---- Date window (start anywhere, include N days)
START_DATE   = date(2026, 6, 1)
NUM_DAYS     = 4
DAY_DATES    = [START_DATE + timedelta(days=i) for i in range(NUM_DAYS)]
# Column labels (change to "%a %m/%d" if you prefer dates in headers)

def _col_label(d: date) -> str:
    try:
        return d.strftime("%A %-m/%-d")     # Mac/Linux
    except ValueError:
        return d.strftime("%A %#m/%#d")     # Windows

COL_LABELS = [_col_label(d) for d in DAY_DATES]


# ---- Output -----#
# --- Specify output directory and filename - creates directory if needed
OUTPUT_DIR   = "output"
OUTPUT_XLSX  = f"StoryPointEL_Listings_{START_DATE}_{NUM_DAYS}_UD_3-3.xlsx"
os.makedirs(OUTPUT_DIR, exist_ok=True)



# ---- Channel-number profile (sports-relevant only)
CHANNEL_MAPS = {
    "StoryPoint EL": {
        # Locals
        "CBS": 3, "NBC": 4, "FOX": 6, "ABC": 7,
        # Sports tier
        "ESPN": 11, "ESPNews": 12, "ESPNU": 13, "ESPN2": 14,
        "FS1": 15, "Tigers TV": 48,
        # Golf scoreboard feeds are handled with the same ESPN scoreboard parser now,
        # but exact round-by-round TV windows may still need a supplemental guide source.
        "Golf": 52,
        # Entertainment nets that carry sports
        "USA": 18, "TNT": 19, "truTV": 20, "TBS": 21,
        # Streaming / other
        "B1G+ APP": 98, "Peacock": 99,
    }
}
ACTIVE_CHANNEL_MAP_NAME = "StoryPoint EL"

# ---- ESPN scoreboard URL overrides
# Most feeds work from site.api.espn.com using the league key directly.
# College softball is an exception discovered in 2025/2026: ESPN exposes it
# under baseball/college-softball on the site.web.api host.
SCOREBOARD_URL_OVERRIDES = {
    "baseball/college-softball": "https://site.web.api.espn.com/apis/site/v2/sports/baseball/college-softball/scoreboard",
}


# ================== SCOREBOARD CATALOG (comment to disable) ==================
SPORTS = [
    ("NFL", ["football/nfl"]),
    ("NCAA FB", ["football/college-football"]),
    ("NBA", ["basketball/nba"]),
    ("NHL", ["hockey/nhl"]),
    ("NCAA Hockey", ["hockey/mens-college-hockey"]),
    # ("NCAA Hockey (M)", ["hockey/mens-college-hockey"]),
    ("MLB", ["baseball/mlb"]),
    
    ("NCAA BB (M)", ["basketball/mens-college-basketball"]),
    ("NCAA BB (W)", ["basketball/womens-college-basketball"]),
    
    ### All Known Available Leagues (uncomment to enable more)
    ("NCAA Baseball", ["baseball/college-baseball"]),
    # ("NCAA Lacrosse(M)", ["lacrosse/mens-college-lacrosse"]),
    ("NCAA Soccer(M)", ["soccer/usa.ncaa.m.1"]),
    # ("Volleyball(M)", ["volleyball/mens-college-volleyball"]),
    # ("NCAA Water Polo(M)", ["waterpolo/mens-college-water-polo"]),
    ("Volleyball", ["volleyball/womens-college-volleyball"]),
    ("NCAA Softball", ["baseball/college-softball"]),
    # ("NCAA Field Hockey", ["fieldhockey/womens-college-field-hockey"]), # # Field Hockey
    # ("NCAA Ice Hockey(W)", ["hockey/womens-college-hockey"]), # # Womens Ice Hockey
    # ("NCAA Lacrosse(W)", ["lacrosse/womens-college-lacrosse"]), # # Lacrosse (W)
    ("NCAA Soccer(W)", ["soccer/usa.ncaa.w.1"]), # # Soccer Women

    # Golf scoreboard feeds return tournament-level events. They usually include
    # networks, but not always exact daily broadcast windows.
    ("PGA", ["golf/pga"]),
    ("LPGA", ["golf/lpga"]),
    ("Champions Tour", ["golf/champions-tour"]),
    # ("LIV Golf", ["golf/liv"]),
    # ("DP World Tour", ["golf/eur"]),

    # ("NCAA Volleyball(W)", ["volleyball/womens-college-volleyball"]), # # Volleyball (W)
    # ("NCAA Water Polo(W)", ["waterpolo/womens-college-water-polo "]), # # Water Polo (W)

    # # Soccer
    ("MLS",   ["soccer/usa.1"]), # MLS USA
    ("EPL",   ["soccer/eng.1"]), # ENGLISH PREMIER
    ("UCL",   ["soccer/uefa.champions"]), # UEFA CHAMPIONS LEAGUE
    # ("La Liga", ["soccer/esp.1"]), # # La Liga
    # ("Bundesliga", ["soccer/ger.1"]), # # Bundesliga
    # ("NWSL",  ["soccer/usa.nwsl"]), # # National Womens NWSL (USA)
    
    # ("Liga MX", ["soccer/mex.1"]), # # Liga MX (Mexico)
]

# ================== TV GUIDE BACKUP / SUPPLEMENT ==================
# ESPN scoreboard feeds are great for games, but they miss some featured TV windows
# and golf is especially messy because the scoreboard is tournament-level.
# This optional scrape adds live-event listings from TV Insider channel pages.
USE_TV_GUIDE_BACKUP = True
TV_GUIDE_SOURCE = "tvinsider"
TV_GUIDE_FILL_MODE = "append_dedupe"  # append guide rows, then drop same channel/date/time/tag duplicates
TV_GUIDE_DEBUG_OUTPUTS = True  # write parsed/kept/rejected guide rows to CSV for troubleshooting
TV_GUIDE_EVENT_STRICTNESS = "live_only"  # "live_only" or "featured_sports"

# For normal channel rows, use guide listings instead of ESPN scoreboard tournament feeds for golf.
# The scoreboard golf items can be useful, but they tend to create all-day/generic rows.
USE_GOLF_SCOREBOARD_FEEDS = False
GOLF_SCOREBOARD_LEAGUES = {"golf/pga", "golf/lpga", "golf/champions-tour", "golf/liv", "golf/eur"}

# TV Insider URL pattern: https://www.tvinsider.com/network/{slug}/schedule/
# Start with the channels that most often carry live sports in this lineup.
TV_GUIDE_CHANNEL_SLUGS = {
    "ESPN": "espn",
    "ESPN2": "espn2",
    "ESPNU": "espnu",
    "ESPNews": "espnews",
    "Golf": "golf-channel",
    "FS1": "fox-sports-1",
    "TBS": "tbs",
    "TNT": "tnt",
    "truTV": "trutv",
    "USA": "usa-network",
    "ABC": "abc",
    "CBS": "cbs",
    "NBC": "nbc",
    "FOX": "fox",
}

# Keep the TV-guide scrape focused on live/featured sports events instead of studio shows,
# documentaries, news/highlights, and routine replays.
TV_GUIDE_EXCLUDE_TITLE_RE = re.compile(
    r"\b("
    r"SportsCenter|SportCenter|Get Up|First Take|Pardon the Interruption|PTI|Around the Horn|"
    r"NBA Today|NFL Live|MLB Tonight|College Football Live|College GameDay|"
    r"Golf Central|Golf Today|The Golf Fix|Ask Rory|5 Clubs|Golf Channel Podcast|"
    r"30 for 30|E60|SC Featured|UFC Unleashed|Poker|BET Live|Daily Wager|"
    r"Championship Update|Postgame|Pregame|Preview|Highlights|The Drop"
    r")\b",
    flags=re.IGNORECASE,
)

TV_GUIDE_INCLUDE_EVENT_RE = re.compile(
    r"("
    r"NFL Football|College Football|NBA Basketball|WNBA Basketball|NHL Hockey|MLB Baseball|"
    r"College Baseball|College Softball|Softball|College Basketball|"
    r"Women.?s College World Series|Womens College World Series|WCWS|College World Series|"
    r"Soccer|UEFA|Premier League|MLS|NWSL|Concacaf|World Cup|"
    r"PGA Tour Golf|LPGA Tour Golf|DP World Tour Golf|Korn Ferry|PGA Tour Champions|College Golf|"
    r"NCAA Men's|NCAA Women.?s|NCAA Womens|National Championship|U\.?S\.? Open|The Open|Masters|PGA Championship|"
    r"Tennis|French Open|Wimbledon|US Open Tennis|Australian Open|"
    r"NASCAR|Formula 1|IndyCar|Auto Racing|Motorsports|Lacrosse|Volleyball"
    r")",
    flags=re.IGNORECASE,
)


# Short tags for the grid
SPORT_TAGS = {
    "football/nfl": "NFL",
    "basketball/nba": "NBA",
    "hockey/nhl": "NHL",
    "baseball/mlb": "MLB",
    "baseball/college-softball": "Softball",
    "football/college-football": "FBS FB",
    "basketball/mens-college-basketball": "M CBB",
    "basketball/womens-college-basketball": "W CBB",
    "soccer/usa.1": "Soccer - MLS",
    "soccer/eng.1": "Soccer - EPL",
    "soccer/uefa.champions": "Soccer - UCL",
    "golf/pga": "PGA Tour",
    "golf/champions-tour": "Champions Tour",
    "golf/lpga": "LPGA",
    "golf/liv": "LIV Golf",
    "golf/eur": "European Tour",
    "racing/f1": "F1",
    "tennis/atp": "ATP",
    "tennis/wta": "WTA",
}

# ---- Favorites config
# Favorite pro teams now support optional row labels and row colors.
# You can still use the old tuple style:
#     ("Boston Bruins", ["hockey/nhl"])
# but the dict style below lets each favorite row carry its own colors.
FAVORITE_PRO_TEAMS = [
    # {
    #     "name": "Boston Bruins",
    #     "leagues": ["hockey/nhl"],
    #     "row_label": "Boston Bruins",
    #     "bg_color": "#FFB81C",
    #     "font_color": "#000000",
    # },
    # {
    #     "name": "Boston Celtics",
    #     "leagues": ["basketball/nba"],
    #     "row_label": "Boston Celtics",
    #     "bg_color": "#007A33",
    #     "font_color": "#FFFFFF",
    # },
    {
        "name": "Boston Red Sox",
        "leagues": ["baseball/mlb"],
        "row_label": "Boston Red Sox",
        "bg_color": "#BD3039",   # Red Sox red
        "font_color": "#FFFFFF",
    },
    {
        "name": "Detroit Tigers",
        "leagues": ["baseball/mlb"],
        "row_label": "Detroit Tigers",
        "bg_color": "#0C2340",   # Tigers navy
        "font_color": "#FFFFFF",
    },
]

# Favorite school: name + leagues to scan (add/remove as needed)
FAVORITE_SCHOOL = {
    "name": "Michigan State",
    "row_label": "MSU \n on B1G+",
    "leagues": [
        "football/college-football",
        "basketball/mens-college-basketball",
        "basketball/womens-college-basketball",
        "hockey/mens-college-hockey",
        "baseball/college-baseball",
        "baseball/college-softball",
        # "soccer/mens-college-soccer",
        # "soccer/womens-college-soccer",
        "volleyball/womens-college-volleyball",
        # add other college leagues here as you discover ESPN keys you care about
    ],
    # special row styling
    "bg_color": "#18453B",   # forest green
    "font_color": "#FFFFFF", # white
}

def _as_favorite_config(fav):
    """
    Normalize favorite-team config so older tuple entries still work.

    Supported:
      ("Boston Red Sox", ["baseball/mlb"])

    Preferred:
      {
          "name": "Boston Red Sox",
          "leagues": ["baseball/mlb"],
          "row_label": "Boston Red Sox",
          "bg_color": "#BD3039",
          "font_color": "#FFFFFF",
      }
    """
    if isinstance(fav, dict):
        return {
            "name": fav["name"],
            "leagues": fav["leagues"],
            "row_label": fav.get("row_label", fav["name"]),
            "bg_color": fav.get("bg_color", "#F5F5F5"),
            "font_color": fav.get("font_color", "#000000"),
        }

    team_name, league_keys = fav
    return {
        "name": team_name,
        "leagues": league_keys,
        "row_label": team_name,
        "bg_color": "#F5F5F5",
        "font_color": "#000000",
    }

# Map ESPN broadcast strings -> canonical channel labels (for the normal rows)
CHANNEL_ALIASES = {
    "abc": "ABC", "abc network": "ABC",
    "fox": "FOX", "fox network": "FOX",
    "cbs": "CBS", "cbs network": "CBS",
    "nbc": "NBC", "nbc network": "NBC", "nbc sports": "NBC", "nbcsn": "NBC",

    "espn": "ESPN", "espn2": "ESPN2", "espnu": "ESPNU",
    "espn news": "ESPNews", "espnnews": "ESPNews", "espnews": "ESPNews",

    "fs1": "FS1", "fox sports 1": "FS1", "fox sports1": "FS1",

    "btn": "Big Ten", "big ten network": "Big Ten",
    "golf channel": "Golf", "golf chnl": "Golf", "golfchannel": "Golf",

    "usa": "USA", "usa network": "USA", "USA Network": "USA",
    "tnt": "TNT", "tnt hd": "TNT",
    "tbs": "TBS",
    "trutv": "truTV",
    "fanduel sn det": "FanDuel",
    "b1g+": "B1G+ APP",
    "peacock": "Peacock",
    "tigers tv": "Tigers TV", "tigerstv": "Tigers TV",
}

## TEST TO EXCLUDE STRERAMING SERVICES
EXCLUDE_STREAMING_KEYS = ("espn+", "espn plus", "espn app", "paramount+", "paramount plus",
                            # "peacock+", "peacock premium"
    )
LOCAL_TZ = ZoneInfo("America/Detroit")

# ========================== IMPORTS ==========================
import re
import math
import pandas as pd
import requests
try:
    from bs4 import BeautifulSoup
except ImportError:
    BeautifulSoup = None
from collections import defaultdict
from urllib.parse import urlencode

# ========================== HELPERS ==========================
def to_local_timestr(iso_str: str):
    if not iso_str:
        return None, None
    ts_utc = pd.to_datetime(iso_str, errors="coerce", utc=True)
    if pd.isna(ts_utc):
        return None, None
    ts_local = ts_utc.tz_convert(LOCAL_TZ).tz_localize(None)
    try:
        hm = ts_local.strftime("%-I:%M%p").lower()
    except ValueError:
        hm = ts_local.strftime("%#I:%M%p").lower()
    return ts_local, hm

def normalize_key(s: str) -> str:
    return re.sub(r"[^a-z0-9+ ]", "", s.lower()).strip()

def _split_broadcast_tokens(raw: str) -> list[str]:
    s = re.sub(r"[\/&]| and ", ",", raw, flags=re.IGNORECASE)
    parts = [p.strip() for p in s.split(",")]
    return [p for p in parts if p]

def normalize_channel_name(name: str) -> str | None:
    if not name:
        return None
    lk = name.lower()
    if any(x in lk for x in EXCLUDE_STREAMING_KEYS):
        return None
    return CHANNEL_ALIASES.get(normalize_key(name))

def sport_is_womens(league_key: str) -> bool:
    return "womens" in league_key or "college-softball" in league_key

def team_label(c: dict) -> str:
    t = (c or {}).get("team", {}) or {}
    rank = (c or {}).get("curatedRank", {}).get("current")
    nm = t.get("displayName") or t.get("shortDisplayName") or t.get("name") or ""
    return (f"#{rank} " if rank and rank != 99 else "") + nm

def build_title(comp: dict, ev: dict | None = None, prefix_womens=False) -> str:
    """Build a compact title for team-vs-team events, with a fallback for golf/event-style feeds."""
    comps = comp.get("competitors", []) or []
    by_side = {c.get("homeAway"): c for c in comps if c.get("homeAway")}
    away = team_label(by_side.get("away", {}))
    home = team_label(by_side.get("home", {}))

    if away and home:
        title = f"{away} at {home}".strip()
    else:
        # Golf and some ESPN event feeds do not have home/away teams.
        title = (comp.get("shortName") or comp.get("name")
                 or (ev or {}).get("shortName") or (ev or {}).get("name")
                 or "Untitled event")
        status_detail = ((comp.get("status") or {}).get("type") or {}).get("shortDetail")
        if status_detail and status_detail.lower() not in title.lower():
            title = f"{title} — {status_detail}"

    if prefix_womens:
        title = f"(W) {title}"
    return title

def _scoreboard_url(league_key: str, day: date) -> str:
    """Return the ESPN scoreboard URL for a league/date, with per-league overrides."""
    ymd = day.strftime("%Y%m%d")
    base_url = SCOREBOARD_URL_OVERRIDES.get(
        league_key,
        f"https://site.api.espn.com/apis/site/v2/sports/{league_key}/scoreboard"
    )
    sep = "&" if "?" in base_url else "?"
    return f"{base_url}{sep}{urlencode({'dates': ymd})}"

def fetch_day_sport_multi(day: date, league_keys: list[str]) -> list[dict]:
    last_err = None
    for k in league_keys:
        url = _scoreboard_url(k, day)
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            return r.json().get("events", []) or []
        except Exception as e:
            last_err = e
            continue
    if last_err:
        print(f"[WARN] {day} {'/'.join(league_keys)} fetch failed: {last_err}")
    return []


def event_occurs_on_day(ev: dict, comp: dict, day: date) -> bool:
    """True when a team game starts that day, or a golf/tournament event spans that day."""
    start_raw = comp.get("date") or comp.get("startDate") or ev.get("date")
    end_raw = comp.get("endDate") or ev.get("endDate")
    start_dt, _ = to_local_timestr(start_raw)
    if start_dt is None:
        return False
    if start_dt.date() == day:
        return True
    if end_raw:
        end_dt, _ = to_local_timestr(end_raw)
        if end_dt is not None:
            return start_dt.date() <= day <= end_dt.date()
    return False

def extract_broadcast_tokens(ev: dict, comp: dict) -> list[str]:
    candidates = []
    for b in (comp.get("broadcasts") or []):
        media = b.get("media") or {}
        for k in ("shortName", "name"):
            if media.get(k): candidates.append(str(media[k]))
        for k in ("shortName", "name"):
            if b.get(k): candidates.append(str(b[k]))
        for n in (b.get("names") or []):
            candidates.append(str(n))
    for gb in (ev.get("geoBroadcasts") or []):
        chan = gb.get("media", {}).get("shortName") or gb.get("media", {}).get("name")
        if chan: candidates.append(str(chan))
    b = comp.get("broadcast") or {}
    if isinstance(b, dict):
        for k in ("shortName", "name"):
            if b.get(k): candidates.append(str(b[k]))
    tokens, seen = [], set()
    for raw in candidates:
        for tok in _split_broadcast_tokens(raw):
            nk = normalize_key(tok)
            if nk and nk not in seen:
                seen.add(nk)
                tokens.append(tok)
    return tokens

def get_event_tag(league_key: str, sport_label: str) -> str:
    tag = SPORT_TAGS.get(league_key)
    if not tag:
        fallback = {
            "NBA":"NBA","NHL":"NHL","NFL":"NFL","MLB":"MLB","CFB":"CFB",
            "MBB":"MBB","WBB":"WBB","MLS":"MLS","EPL":"EPL","UCL":"UCL",
            "PGA":"PGA","LPGA":"LPGA","LIV":"LIV","DPW":"DPW"
        }.get(sport_label, "SPORT")
        tag = fallback
    return f"({tag})"

def _break_after_at(title: str) -> str:
    return re.sub(r"\s+(at|vs\.?|v\.)\s+", r" \1\n", title, count=1, flags=re.IGNORECASE)

def _comp_has_team(comp: dict, needle: str) -> bool:
    """case-insensitive contains on team display names."""
    if not needle:
        return False
    needle = needle.lower()
    for c in (comp.get("competitors") or []):
        t = (c or {}).get("team", {}) or {}
        for f in ("displayName", "shortDisplayName", "name"):
            val = (t.get(f) or "").lower()
            if val and needle in val:
                return True
    return False


# ========================== TV GUIDE BACKUP HELPERS ==========================
def _tvguide_header_for_date(d: date) -> str:
    return f"{d.strftime('%A')}, {d.strftime('%B')} {d.day}"

TV_GUIDE_DATE_HEADER_MAP = {_tvguide_header_for_date(d): d for d in DAY_DATES}
TV_GUIDE_TIME_RE = re.compile(r"^(\d{1,2}:\d{2})\s*([AP]M)(?:\s*(ET|EDT|EST))?$", flags=re.IGNORECASE)
TV_GUIDE_COMPACT_TIME_RE = re.compile(r"^(\d{1,2}:\d{2}\s*[AP]M)\s+(.+)$", flags=re.IGNORECASE)
TV_GUIDE_DATE_HEADER_RE = re.compile(
    r"^(?:#+\s*)?(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),\s+([A-Za-z]+)\s+(\d{1,2})(?:,\s*(\d{4}))?\b",
    flags=re.IGNORECASE,
)
TV_GUIDE_MONTH_LOOKUP = {m.lower(): i for i, m in enumerate([
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
], start=1)}

TV_GUIDE_PARSED_ITEMS = []
TV_GUIDE_REJECTED_ITEMS = []
TV_GUIDE_KEPT_ITEMS = []


def _clean_tvguide_line(line: str) -> str:
    line = re.sub(r"\s+", " ", str(line or "")).strip()
    # TV Insider sometimes exposes image alt text and nav labels in parsed text.
    if line.lower().startswith("image:"):
        return ""
    # Strip markdown-ish heading prefixes if the page is returned in a rendered-text style.
    line = re.sub(r"^#{1,6}\s*", "", line).strip()
    return line


def _strip_tvguide_html(html: str) -> list[str]:
    if BeautifulSoup is not None:
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script", "style", "noscript", "svg"]):
            tag.decompose()

        # Put block-ish elements on their own lines. TV Insider sometimes compresses
        # schedule-card links into one text node like:
        # "12:00 PM 2026 Women’s College World Series New Sports • From ..."
        for tag in soup.find_all(["br", "p", "div", "li", "time", "h1", "h2", "h3", "h4", "h5", "h6"]):
            tag.insert_before("\n")
            tag.insert_after("\n")
        raw_lines = soup.get_text("\n").splitlines()
    else:
        # Fallback if beautifulsoup4 is not installed. It is rougher, but good enough
        # to preserve date/time/title lines on TV Insider pages.
        cleaned = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.I | re.S)
        cleaned = re.sub(r"<br\s*/?>", "\n", cleaned, flags=re.I)
        cleaned = re.sub(r"</(h\d|p|div|li|time|span|a)>", "\n", cleaned, flags=re.I)
        cleaned = re.sub(r"<[^>]+>", " ", cleaned)
        raw_lines = cleaned.splitlines()
    return [ln for ln in (_clean_tvguide_line(x) for x in raw_lines) if ln]


def _tvguide_datetime(show_date: date, time_text: str) -> tuple[datetime | None, str | None]:
    m = TV_GUIDE_TIME_RE.match(time_text.strip())
    if not m:
        return None, None
    compact = f"{m.group(1)} {m.group(2).upper()}"
    try:
        dt = datetime.strptime(f"{show_date.isoformat()} {compact}", "%Y-%m-%d %I:%M %p")
    except ValueError:
        return None, None
    try:
        hm = dt.strftime("%-I:%M%p").lower()
    except ValueError:
        hm = dt.strftime("%#I:%M%p").lower()
    return dt, hm


def _is_tvguide_time_line(line: str) -> bool:
    return bool(TV_GUIDE_TIME_RE.match(line.strip()))


def _parse_tvguide_date_header(line: str) -> date | None:
    m = TV_GUIDE_DATE_HEADER_RE.match(_clean_tvguide_line(line))
    if not m:
        return None
    month_num = TV_GUIDE_MONTH_LOOKUP.get(m.group(2).lower())
    if not month_num:
        return None
    day_num = int(m.group(3))
    explicit_year = int(m.group(4)) if m.group(4) else None

    # TV Insider headers usually omit the year. Try the current schedule window year
    # and neighboring years so Dec/Jan windows do not break.
    candidate_years = [explicit_year] if explicit_year else [START_DATE.year, START_DATE.year - 1, START_DATE.year + 1]
    for y in candidate_years:
        try:
            cand = date(y, month_num, day_num)
        except ValueError:
            continue
        if cand in DAY_DATES:
            return cand
    return None


def _is_tvguide_date_header(line: str) -> bool:
    return _parse_tvguide_date_header(line) is not None


def _clean_tvguide_title(title: str) -> str:
    title = _clean_tvguide_line(title)
    title = re.sub(r"^Stream\s+", "", title, flags=re.I).strip()
    title = re.sub(r"\s+", " ", title).strip()
    title = re.sub(r"\s+(New|Live)$", "", title, flags=re.I).strip()
    return title


def _clean_tvguide_detail_lines(lines: list[str]) -> str:
    keep = []
    for ln in lines:
        ln = _clean_tvguide_line(ln)
        if not ln:
            continue
        low = ln.lower()
        if low in {"sports", "new", "live", "replay", "replays", "live & upcoming", "on-air schedule"}:
            continue
        if low.startswith("sports •") or low.startswith("season ") or low.startswith("episode "):
            continue
        if re.match(r"^(tv|tv-g|tvg|tv-pg|tv-14|cc)$", low):
            continue
        if "click a program" in low or "complete schedule" in low or "change channel" in low:
            continue
        keep.append(ln)
    return " ".join(keep).strip()


def _split_compact_tvguide_listing(line: str) -> tuple[str | None, str | None, str]:
    """
    Handle TV Insider rows that arrive as one compressed text line, e.g.
    '12:00 PM 2026 Women’s College World Series New Sports • From Devon Park...'
    Returns: time_text, title, description
    """
    m = TV_GUIDE_COMPACT_TIME_RE.match(_clean_tvguide_line(line))
    if not m:
        return None, None, ""
    time_text = m.group(1)
    rest = m.group(2).strip()

    title_part = rest
    desc_part = ""
    # Most useful TV Insider compressed rows have 'Sports •' between title/meta and description.
    parts = re.split(r"\bSports\s*•\s*", rest, maxsplit=1, flags=re.I)
    if len(parts) == 2:
        title_part, desc_part = parts[0].strip(), parts[1].strip()

    # Remove 'New'/'Live' and trailing years from the title side.
    title_part = re.sub(r"\s+(New|Live)\b.*$", "", title_part, flags=re.I).strip()
    title_part = re.sub(r"\s+\d{4}$", "", title_part).strip()
    title_part = _clean_tvguide_title(title_part)
    return time_text, title_part, _clean_tvguide_detail_lines([desc_part])


def parse_tvinsider_schedule(channel: str, slug: str) -> list[dict]:
    url = f"https://www.tvinsider.com/network/{slug}/schedule/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }
    try:
        r = requests.get(url, headers=headers, timeout=25)
        r.raise_for_status()
    except Exception as e:
        print(f"[WARN] TV guide fetch failed for {channel} ({url}): {e}")
        return []

    lines = _strip_tvguide_html(r.text)
    out = []
    current_date = None
    i = 0
    while i < len(lines):
        line = lines[i]
        parsed_date = _parse_tvguide_date_header(line)
        if parsed_date is not None:
            current_date = parsed_date
            i += 1
            continue

        if current_date in DAY_DATES:
            # Case A: normal multiline listing; line is just the time.
            if _is_tvguide_time_line(line):
                time_text = line
                j = i + 1
                while j < len(lines) and not lines[j]:
                    j += 1
                if j >= len(lines):
                    break
                title = _clean_tvguide_title(lines[j])
                detail_lines = []
                k = j + 1
                while k < len(lines):
                    nxt = lines[k]
                    if _parse_tvguide_date_header(nxt) is not None or _is_tvguide_time_line(nxt):
                        break
                    # Stop before another compressed same-day listing.
                    if TV_GUIDE_COMPACT_TIME_RE.match(nxt):
                        break
                    detail_lines.append(nxt)
                    k += 1

                start_dt, hm = _tvguide_datetime(current_date, time_text)
                if start_dt is not None:
                    out.append({
                        "source": "tvguide",
                        "channel": channel,
                        "date": current_date,
                        "start_dt": start_dt,
                        "time_str": hm,
                        "raw_time": time_text,
                        "title": title,
                        "description": _clean_tvguide_detail_lines(detail_lines),
                        "url": url,
                    })
                i = k
                continue

            # Case B: compressed listing; line starts with time + title + metadata.
            time_text, title, desc = _split_compact_tvguide_listing(line)
            if time_text and title:
                start_dt, hm = _tvguide_datetime(current_date, time_text)
                if start_dt is not None:
                    out.append({
                        "source": "tvguide",
                        "channel": channel,
                        "date": current_date,
                        "start_dt": start_dt,
                        "time_str": hm,
                        "raw_time": time_text,
                        "title": title,
                        "description": desc,
                        "url": url,
                    })
                i += 1
                continue

        i += 1
    return out


def infer_tvguide_tag(title: str, desc: str = "") -> str:
    text = f"{title} {desc}".lower()
    text_ascii = (
        text.replace("’", "'")
            .replace("women’s", "women's")
            .replace("womens", "women's")
            .replace("u.s.", "us")
    )
    if "women's college world series" in text_ascii or "wcws" in text_ascii:
        return "(Softball)"
    if "softball" in text_ascii:
        return "(Softball)"
    if "college world series" in text_ascii or "college baseball" in text_ascii:
        return "(NCAA Baseball)"
    if "mlb" in text_ascii or "baseball" in text_ascii:
        return "(MLB)"
    if "nba finals" in text_ascii or "nba" in text_ascii:
        return "(NBA)"
    if "wnba" in text_ascii:
        return "(WNBA)"
    if "stanley cup" in text_ascii or "nhl" in text_ascii or "hockey" in text_ascii:
        return "(NHL)"
    if "college football" in text_ascii:
        return "(FBS FB)"
    if "nfl" in text_ascii or "football" in text_ascii:
        return "(NFL)"
    if "college golf" in text_ascii:
        return "(College Golf)"
    if "lpga" in text_ascii or "women's open" in text_ascii:
        return "(LPGA)"
    if "champions" in text_ascii and "golf" in text_ascii:
        return "(Champions Tour)"
    if "dp world" in text_ascii:
        return "(DPW)"
    if "korn ferry" in text_ascii:
        return "(KFT)"
    if "pga tour golf" in text_ascii or "memorial tournament" in text_ascii or "masters" in text_ascii or "pga championship" in text_ascii or "us open" in text_ascii or "the open" in text_ascii:
        return "(PGA Tour)"
    if "soccer" in text_ascii or "uefa" in text_ascii or "premier league" in text_ascii or "mls" in text_ascii:
        return "(Soccer)"
    if "tennis" in text_ascii or "french open" in text_ascii or "wimbledon" in text_ascii:
        return "(Tennis)"
    if "nascar" in text_ascii or "formula 1" in text_ascii or "indycar" in text_ascii or "auto racing" in text_ascii:
        return "(Racing)"
    if "volleyball" in text_ascii:
        return "(Volleyball)"
    return "(TV)"


def is_live_tvguide_event(item: dict) -> tuple[bool, str]:
    title = item.get("title") or ""
    desc = item.get("description") or ""
    text = f"{title} {desc}"

    if TV_GUIDE_EXCLUDE_TITLE_RE.search(title):
        return False, "excluded_title"

    # Strict mode is for actual games/events, not studio shows or shoulder programming.
    if TV_GUIDE_EVENT_STRICTNESS == "live_only":
        if re.search(r"\b(Pregame|Postgame|Preview|Highlights|Live From|Tip-Off|Championship Update)\b", title, flags=re.I):
            return False, "strict_studio_or_shoulder"

    tag = infer_tvguide_tag(title, desc)
    if tag != "(TV)":
        return True, "tag_inferred"

    if TV_GUIDE_INCLUDE_EVENT_RE.search(text):
        return True, "include_regex"

    return False, "no_live_event_match"


def _shorten_for_grid(text: str, max_chars: int = 105) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0].rstrip(".,;:") + "…"


def tvguide_display_title(item: dict) -> str:
    title = item.get("title") or "Untitled TV event"
    desc = item.get("description") or ""
    # For generic listings like "PGA Tour Golf" or "College Softball", the useful
    # tournament/game detail usually lives in the description.
    if desc and title.lower() not in desc.lower():
        return f"{title} — {_shorten_for_grid(desc)}"
    return title


def build_tvguide_rows() -> list[dict]:
    global TV_GUIDE_PARSED_ITEMS, TV_GUIDE_REJECTED_ITEMS, TV_GUIDE_KEPT_ITEMS
    if not USE_TV_GUIDE_BACKUP:
        return []
    active_channels = CHANNEL_MAPS[ACTIVE_CHANNEL_MAP_NAME]
    guide_rows = []
    TV_GUIDE_PARSED_ITEMS = []
    TV_GUIDE_REJECTED_ITEMS = []
    TV_GUIDE_KEPT_ITEMS = []

    for channel, slug in TV_GUIDE_CHANNEL_SLUGS.items():
        if channel not in active_channels:
            continue
        parsed_items = parse_tvinsider_schedule(channel, slug)
        for item in parsed_items:
            TV_GUIDE_PARSED_ITEMS.append(item.copy())
            keep, reason = is_live_tvguide_event(item)
            diagnostic = item.copy()
            diagnostic["decision_reason"] = reason
            diagnostic["inferred_tag"] = infer_tvguide_tag(item.get("title", ""), item.get("description", ""))
            if not keep:
                TV_GUIDE_REJECTED_ITEMS.append(diagnostic)
                continue
            tag = diagnostic["inferred_tag"]
            row = {
                "col_label": _col_label(item["date"]),
                "date": item["date"],
                "time_str": item["time_str"],
                "start_dt": item["start_dt"],
                "sport": "TV Guide",
                "league_key": "tvguide",
                "tag": tag,
                "channel": channel,
                "channel_num": active_channels[channel],
                "title": tvguide_display_title(item),
                "fav_row": None,
                "source": "tvguide",
            }
            guide_rows.append(row)
            kept_diag = diagnostic.copy()
            kept_diag.update(row)
            TV_GUIDE_KEPT_ITEMS.append(kept_diag)

    print(
        f"TV guide backup parsed {len(TV_GUIDE_PARSED_ITEMS)} total listings; "
        f"kept {len(TV_GUIDE_KEPT_ITEMS)} candidate live-event rows; "
        f"rejected {len(TV_GUIDE_REJECTED_ITEMS)}"
    )

    if TV_GUIDE_DEBUG_OUTPUTS:
        try:
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            if TV_GUIDE_PARSED_ITEMS:
                pd.DataFrame(TV_GUIDE_PARSED_ITEMS).to_csv(os.path.join(OUTPUT_DIR, "tvguide_parsed_all.csv"), index=False)
            if TV_GUIDE_KEPT_ITEMS:
                pd.DataFrame(TV_GUIDE_KEPT_ITEMS).to_csv(os.path.join(OUTPUT_DIR, "tvguide_kept.csv"), index=False)
            if TV_GUIDE_REJECTED_ITEMS:
                pd.DataFrame(TV_GUIDE_REJECTED_ITEMS).to_csv(os.path.join(OUTPUT_DIR, "tvguide_rejected.csv"), index=False)
        except Exception as e:
            print(f"[WARN] Could not write TV guide debug CSVs: {e}")

    if guide_rows:
        guide_summary = pd.DataFrame(guide_rows).groupby(["channel", "tag"]).size().reset_index(name="rows")
        print("TV guide kept rows by channel/tag:")
        print(guide_summary.to_string(index=False))

    return guide_rows


def should_skip_scoreboard_for_normal_grid(league_keys: list[str]) -> bool:
    if USE_TV_GUIDE_BACKUP and not USE_GOLF_SCOREBOARD_FEEDS:
        return any(k in GOLF_SCOREBOARD_LEAGUES for k in league_keys)
    return False

# ========================== GATHER EVENTS ==========================
active_map = CHANNEL_MAPS[ACTIVE_CHANNEL_MAP_NAME]
unknown_broadcasts = defaultdict(int)
rows = []

# Normal channel-based events (filter to active_map)
for day in DAY_DATES:
    for sport_label, league_keys in SPORTS:
        if should_skip_scoreboard_for_normal_grid(league_keys):
            continue
        events = fetch_day_sport_multi(day, league_keys)
        if not events:
            continue
        for ev in events:
            for comp in (ev.get("competitions") or []):
                local_dt, hm = to_local_timestr(comp.get("date") or ev.get("date"))
                if local_dt is None or not event_occurs_on_day(ev, comp, day):
                    continue
                event_day = local_dt.date() if local_dt.date() == day else day
                col_label = _col_label(event_day)
                if local_dt.date() != day:
                    hm = "TBD"
                    sort_dt = datetime.combine(event_day, datetime.min.time())
                else:
                    sort_dt = local_dt

                # broadcasts -> channels
                channel_hits = set()
                for raw in extract_broadcast_tokens(ev, comp):
                    canon = normalize_channel_name(raw)
                    if canon:
                        channel_hits.add(canon)
                    else:
                        key = normalize_key(raw)
                        if key and not any(x in key for x in EXCLUDE_STREAMING_KEYS):
                            unknown_broadcasts[raw] += 1

                channel_hits = [c for c in channel_hits if c in active_map]
                if not channel_hits:
                    continue

                title = build_title(comp, ev=ev, prefix_womens=sport_is_womens(league_keys[0]))
                tag_key = next((k for k in league_keys if k in SPORT_TAGS), None)
                tag = f"({SPORT_TAGS.get(tag_key, sport_label)})"

                for ch in channel_hits:
                    rows.append({
                        "col_label": col_label,
                        "date": event_day,
                        "time_str": hm,
                        "start_dt": sort_dt,
                        "sport": sport_label,
                        "league_key": tag_key or league_keys[0],
                        "tag": tag,
                        "channel": ch,
                        "channel_num": active_map.get(ch),
                        "title": title,
                        "fav_row": None,   # normal grid
                        "source": "scoreboard",
                    })

# TV guide backup rows. These supplement the ESPN scoreboard output and are
# especially useful for Golf Channel / featured ESPN windows that do not map cleanly
# from the scoreboard feeds.
rows.extend(build_tvguide_rows())

# Favorite pro teams (ignore channel filters; force into special rows)
FAV_PRO_ROWS = []     # [(row_num, row_label)]
FAV_ROW_STYLES = {}   # (row_num, row_label) -> {"bg_color": ..., "font_color": ..., "name": ...}

for i, fav in enumerate(FAVORITE_PRO_TEAMS):
    fav_cfg = _as_favorite_config(fav)
    team_name = fav_cfg["name"]
    league_keys = fav_cfg["leagues"]

    row_num = max(active_map.values()) + 10 + i     # place after normal channels in printed row list
    row_label = fav_cfg["row_label"]

    FAV_PRO_ROWS.append((row_num, row_label))
    FAV_ROW_STYLES[(row_num, row_label)] = {
        "name": team_name,
        "bg_color": fav_cfg["bg_color"],
        "font_color": fav_cfg["font_color"],
    }

    for day in DAY_DATES:
        events = fetch_day_sport_multi(day, league_keys)
        if not events:
            continue
        for ev in events:
            for comp in (ev.get("competitions") or []):
                if not _comp_has_team(comp, team_name):
                    continue
                local_dt, hm = to_local_timestr(comp.get("date") or ev.get("date"))
                if local_dt is None or local_dt.date() not in DAY_DATES:
                    continue
                col_label = _col_label(local_dt.date())
                title = build_title(comp, ev=ev, prefix_womens=sport_is_womens(league_keys[0]))
                tag = get_event_tag(next((k for k in league_keys if k in SPORT_TAGS), league_keys[0]), sport_label=team_name)
                rows.append({
                    "col_label": col_label,
                    "date": local_dt.date(),
                    "time_str": hm,
                    "start_dt": local_dt,
                    "sport": team_name,           # display purpose
                    "league_key": league_keys[0], # best-effort
                    "tag": tag,
                    "channel": row_label,
                    "channel_num": row_num,
                    "title": title,
                    "fav_row": team_name,         # mark favorite row by team name
                    "source": "favorite",
                })

# Favorite school (all listed college leagues; special styling)
SCHOOL_ROW_NUM   = 100 # sort after normal channels
SCHOOL_ROW_LABEL = FAVORITE_SCHOOL.get("row_label", FAVORITE_SCHOOL["name"])

FAV_ROW_STYLES[(SCHOOL_ROW_NUM, SCHOOL_ROW_LABEL)] = {
    "name": FAVORITE_SCHOOL["name"],
    "bg_color": FAVORITE_SCHOOL["bg_color"],
    "font_color": FAVORITE_SCHOOL["font_color"],
}

for day in DAY_DATES:
    for k in FAVORITE_SCHOOL["leagues"]:
        events = fetch_day_sport_multi(day, [k])
        if not events:
            continue
        for ev in events:
            for comp in (ev.get("competitions") or []):
                if not _comp_has_team(comp, FAVORITE_SCHOOL["name"]):
                    continue
                local_dt, hm = to_local_timestr(comp.get("date") or ev.get("date"))
                if local_dt is None or local_dt.date() not in DAY_DATES:
                    continue
                col_label = _col_label(local_dt.date())
                title = build_title(comp, ev=ev, prefix_womens=sport_is_womens(k))
                tag = f"({SPORT_TAGS.get(k, FAVORITE_SCHOOL['name'])})"
                rows.append({
                    "col_label": col_label,
                    "date": local_dt.date(),
                    "time_str": hm,
                    "start_dt": local_dt,
                    "sport": FAVORITE_SCHOOL["name"],
                    "league_key": k,
                    "tag": tag,
                    "channel": SCHOOL_ROW_LABEL,
                    "channel_num": SCHOOL_ROW_NUM,
                    "title": title,
                    "fav_row": FAVORITE_SCHOOL["name"],  # mark favorite row by team name
                    "source": "favorite",
                })


# ---------------- DataFrame ----------------
df = pd.DataFrame(rows)
if df.empty:
    print("No events matched. Check dates, SPORTS, favorites, and mapping.")
else:
    if "source" not in df.columns:
        df["source"] = "scoreboard"

    # When TV-guide rows duplicate a scoreboard row at the same channel/date/time/sport,
    # keep the scoreboard version because it usually has the better team-vs-team title.
    # Golf scoreboard rows are suppressed above by default, so Golf Channel guide rows survive.
    source_priority = {"favorite": 0, "scoreboard": 1, "tvguide": 2}
    df["_source_priority"] = df["source"].map(source_priority).fillna(9)
    before_dedupe = len(df)
    df = (
        df.sort_values(["_source_priority", "date", "channel_num", "start_dt", "title"])
          .drop_duplicates(["date", "channel_num", "channel", "start_dt", "tag"], keep="first")
          .drop(columns=["_source_priority"])
          .sort_values(["date", "channel_num", "start_dt", "title"])
          .reset_index(drop=True)
    )
    dropped = before_dedupe - len(df)
    if dropped:
        print(f"Dropped {dropped} duplicate TV/scoreboard rows")

    df.to_csv(os.path.join(OUTPUT_DIR, "events_flat_storypoint.csv"), index=False)
    print("Wrote flat CSV with", len(df), "rows")

# ========================== EXCEL WRITER (rich text, borders, alt shading + fav rows) ==========================
if not df.empty:
    df["col_label"] = pd.Categorical(df["col_label"], categories=COL_LABELS, ordered=True)

    # Per-cell: (channel_num, channel, col_label) -> [(time, tag, title, fav_row_flag)]
    events_by_cell = defaultdict(list)
    for r in df.itertuples():
        events_by_cell[(r.channel_num, r.channel, r.col_label)].append((r.time_str, r.tag, r.title, r.fav_row))

    # Row order: favorites first, then normal channels by number
    active_map = CHANNEL_MAPS[ACTIVE_CHANNEL_MAP_NAME]
    normal_rows = sorted([(num, ch) for ch, num in active_map.items()], key=lambda x: (x[0], str(x[1])))
    fav_rows = [(SCHOOL_ROW_NUM, SCHOOL_ROW_LABEL)] + FAV_PRO_ROWS
    row_index = fav_rows + normal_rows

    # title/subheader
    def fmt(d: date) -> str: return d.strftime("%B %d, %Y")
    TITLE_TEXT   = f"{fmt(START_DATE)} – {fmt(DAY_DATES[-1])}"
    SUBHEAD_TEXT = f"StoryPoint – East Lansing | Created {datetime.now(LOCAL_TZ).strftime('%B %d, %Y')} by J.Smith"

    # column positions
    first_col     = 1
    num_col       = first_col
    chan_col      = first_col + 1
    first_day_col = chan_col + 1
    last_day_col  = chan_col + len(COL_LABELS)

    out_path = os.path.join(OUTPUT_DIR, OUTPUT_XLSX)
    with pd.ExcelWriter(out_path, engine="xlsxwriter") as xw:
        wb  = xw.book
        ws  = wb.add_worksheet("Week")

        # Per-sport font colors
        SPORT_STYLE = {
            "NFL": "#B22222", "NBA": "#5C2E91", "NHL": "#1F4E79", "MLB": "#0A2463",
            "SOFTBALL": "#BE123C", "Softball": "#BE123C",
            "FBS FB": "#0B6E4F", "CFB": "#0B6E4F", "M CBB": "#D97706", "W CBB": "#C026D3",
            "MBB": "#D97706", "WBB": "#C026D3",
            "MLS": "#0B7285", "EPL": "#1D4ED8", "UCL": "#6D28D9", "MCH": "#065F46",
            "SOCCER - MLS": "#0B7285", "SOCCER - EPL": "#1D4ED8", "SOCCER - UCL": "#6D28D9",
            "PGA": "#047857", "PGA TOUR": "#047857", "LPGA": "#BE185D", "LIV": "#111827", "DPW": "#B45309", "KFT": "#374151",
            "NCAA BASEBALL": "#92400E", "WNBA": "#C026D3", "SOCCER": "#0B7285", "TENNIS": "#166534",
            "RACING": "#7C2D12", "VOLLEYBALL": "#9333EA", "TV": "#374151",
        }

        # Base formats
        title_fmt = wb.add_format({"bold": True, "align": "left", "valign": "vcenter", "font_size": 36 })
        sub_fmt   = wb.add_format({"italic": True, "align": "left", "valign": "vcenter", "font_size": 12, "font_color": "#555555"})

        header_fmt = wb.add_format({"bold": True, "align": "center", "valign": "vcenter", "font_size": 14, "border": 1})

        num_fmt         = wb.add_format({"bold": True, "align": "center", "valign": "vcenter", "font_size": 24, "bottom": 1})
        chan_fmt        = wb.add_format({"bold": True, "align": "center",  "valign": "vcenter", "font_size": 22, "bottom": 1})
        num_fmt_shaded  = wb.add_format({"bold": True, "align": "center", "valign": "vcenter", "font_size": 24, "bottom": 1, "bg_color": "#F5F5F5"})
        chan_fmt_shaded = wb.add_format({"bold": True, "align": "center",  "valign": "vcenter", "font_size": 22, "bottom": 1, "bg_color": "#F5F5F5"})

        base_cell_fmt        = wb.add_format({"text_wrap": True, "valign": "vcenter", "bottom": 1})
        base_cell_fmt_shaded = wb.add_format({"text_wrap": True, "valign": "vcenter", "bottom": 1, "bg_color": "#F5F5F5"})

        # Favorite-row formats are generated from FAV_ROW_STYLES so every favorite
        # can use its own team colors, not just MSU.
        _favorite_format_cache = {}

        def _get_favorite_formats(fav_style: dict):
            bg = fav_style.get("bg_color", "#F5F5F5")
            fg = fav_style.get("font_color", "#000000")
            key = (bg, fg)
            if key in _favorite_format_cache:
                return _favorite_format_cache[key]

            cell_fmt = wb.add_format({
                "text_wrap": True,
                "valign": "vcenter",
                "bottom": 1,
                "bg_color": bg,
                "font_color": fg,
            })
            time_fmt = wb.add_format({
                "bold": True,
                "font_size": 16,
                "font_color": fg,
            })
            teams_fmt = wb.add_format({
                "font_size": 14,
                "font_color": fg,
            })
            num_hdr_fmt = wb.add_format({
                "bold": True,
                "align": "center",
                "valign": "vcenter",
                "font_size": 24,
                "bottom": 1,
                "bg_color": bg,
                "font_color": fg,
            })
            chan_hdr_fmt = wb.add_format({
                "bold": True,
                "align": "center",
                "valign": "vcenter",
                "font_size": 22,
                "bottom": 1,
                "bg_color": bg,
                "font_color": fg,
            })

            _favorite_format_cache[key] = (cell_fmt, time_fmt, teams_fmt, num_hdr_fmt, chan_hdr_fmt)
            return _favorite_format_cache[key]

        # Titles
        ws.merge_range(0, num_col, 0, last_day_col, f"Live Sports on TV - Week of {fmt(START_DATE)}", title_fmt)
        ws.merge_range(1, num_col, 1, last_day_col, SUBHEAD_TEXT, sub_fmt)

        # Column headers
        ws.write(2, num_col,  "#", num_fmt)
        ws.write(2, chan_col, "Channel", chan_fmt)
        for c, lbl in enumerate(COL_LABELS):
            ws.write(2, first_day_col + c, lbl, header_fmt)

        # Widths
        ws.set_column(num_col,  num_col, 12)
        ws.set_column(chan_col, chan_col, 20)   # wider for "Streaming – …"
        ws.set_column(first_day_col, last_day_col, 36)

        # Helpers
        def _tag_key(tag_text: str) -> str:
            return (tag_text or "").strip().strip("()").upper()

        _format_cache = {}
        def _get_event_formats(tag: str, shaded: bool):
            key = (tag, shaded)
            if key in _format_cache:
                return _format_cache[key]
            color = SPORT_STYLE.get(tag)
            time_kwargs = {"bold": True, "font_size": 16}
            team_kwargs = {"font_size": 14}
            if color:
                time_kwargs["font_color"] = color
                team_kwargs["font_color"] = color
            tt_fmt = wb.add_format(time_kwargs)
            tm_fmt = wb.add_format(team_kwargs)
            cell_fmt = base_cell_fmt_shaded if shaded else base_cell_fmt
            _format_cache[key] = (tt_fmt, tm_fmt, cell_fmt)
            return _format_cache[key]

        def write_rich_cell(row_idx: int, col_idx: int, evts, shaded: bool, fav_style: dict | None):
            """
            evts: list of (time_str, tag_text, title, fav_row_flag)
            fav_style: None for normal rows, or {"bg_color": ..., "font_color": ...}
            """
            if fav_style:
                fav_cell_fmt, fav_time_fmt, fav_teams_fmt, _, _ = _get_favorite_formats(fav_style)
            else:
                fav_cell_fmt = fav_time_fmt = fav_teams_fmt = None

            if not evts:
                # choose correct base format
                base_fmt = (
                    fav_cell_fmt if fav_style
                    else (base_cell_fmt_shaded if shaded else base_cell_fmt)
                )
                ws.write(row_idx, col_idx, "", base_fmt)
                return

            parts = []
            for i, (time_str, tag_text, title, _) in enumerate(evts):
                tag = _tag_key(tag_text)
                if fav_style:
                    # use this favorite team's row colors
                    parts.extend([fav_time_fmt, f"{time_str}: ({tag})"])
                    parts.append("\n")
                    parts.extend([fav_teams_fmt, _break_after_at(title)])
                else:
                    tt_fmt, tm_fmt, _ = _get_event_formats(tag, shaded)
                    parts.extend([tt_fmt, f"{time_str}: ({tag})"])
                    parts.append("\n")
                    parts.extend([tm_fmt, _break_after_at(title)])
                if i != len(evts) - 1:
                    parts.append("\n")

            # trailing format controls background/wrap
            trailing_fmt = (
                fav_cell_fmt if fav_style
                else (base_cell_fmt_shaded if shaded else base_cell_fmt)
            )
            parts.append(trailing_fmt)
            ws.write_rich_string(row_idx, col_idx, *parts)

        # Body with alternating shading for normal rows; favorite rows use team colors
        current_row = 3
        for idx, (num, ch) in enumerate(row_index):
            fav_style = FAV_ROW_STYLES.get((num, ch))
            if fav_style:
                shaded = False     # favorite rows have their own bg
                _, _, _, row_num_fmt, row_chan_fmt = _get_favorite_formats(fav_style)
            else:
                shaded = (idx % 2 == 1)
                row_num_fmt  = num_fmt_shaded  if shaded else num_fmt
                row_chan_fmt = chan_fmt_shaded if shaded else chan_fmt

            # Row header cells
            try:
                ws.write(current_row, num_col, int(num) if num is not None else "", row_num_fmt)
            except Exception:
                ws.write(current_row, num_col, "", row_num_fmt)
            ws.write(current_row, chan_col, str(ch), row_chan_fmt)

            # Day cells
            for c, lbl in enumerate(COL_LABELS):
                evts = events_by_cell.get((num, ch, lbl), [])
                write_rich_cell(current_row, first_day_col + c, evts, shaded, fav_style)

            current_row += 1

                # Footer or summary rows could be added here if needed

        # ---- Page setup / print settings ----
        from xlsxwriter.utility import xl_range

        last_row = current_row - 1              # last row we wrote
        first_row = 0                           # include title/subheader
        first_col = num_col                     # start at the "#" column
        last_col  = last_day_col                # last day column

        # Page orientation and paper
        ws.set_landscape()          # Landscape
        ws.set_paper(1)             # 1 = Letter (8.5" x 11"). Use 9 for A4.

        # Margins Set to Max
        
        ws.set_margins(left=0.2, right=0.2, top=0.2, bottom=0.2)

        # Center on page (nice for single sheet handouts)
        ws.center_horizontally()
        ws.center_vertically()

        # Fit to a single printed page (1 page wide × 1 page tall)
        ws.fit_to_pages(1, 1)

        # Define the print area so the sheet opens ready to print one page
        ws.print_area(first_row, first_col, last_row, last_col)

        # Optional: repeat header row (the row with "# / Channel / dates") on each printed page
        # (harmless even when we fit to one page)
        ws.repeat_rows(2, 2)  # zero-based row index; your headers are on row 2

        # Optional: show/hide gridlines in print (2 = hide on screen & print)
        ws.hide_gridlines(2)

        # Optional: header/footer
        ws.set_header('&L&"Calibri,Bold"&12StoryPoint – East Lansing'
                    '&R&"Calibri"&10Printed &D')
        # ws.set_footer('&CPage &P of &N')



    print("Wrote workbook:", out_path)

# ========================== UNKNOWN BROADCASTS LOG ==========================
if unknown_broadcasts:
    pd.DataFrame(
        sorted(unknown_broadcasts.items(), key=lambda x: (-x[1], x[0])),
        columns=["raw_broadcast_string","count"]
    ).to_csv(os.path.join(OUTPUT_DIR, "unknown_broadcasts.csv"), index=False)
    print("Wrote unknown_broadcasts.csv (consider adding aliases).")


[WARN] TV guide fetch failed for ESPN (https://www.tvinsider.com/network/espn/schedule/): 403 Client Error: Forbidden for url: https://www.tvinsider.com/network/espn/schedule/
[WARN] TV guide fetch failed for ESPN2 (https://www.tvinsider.com/network/espn2/schedule/): 403 Client Error: Forbidden for url: https://www.tvinsider.com/network/espn2/schedule/
[WARN] TV guide fetch failed for ESPNU (https://www.tvinsider.com/network/espnu/schedule/): 403 Client Error: Forbidden for url: https://www.tvinsider.com/network/espnu/schedule/
[WARN] TV guide fetch failed for ESPNews (https://www.tvinsider.com/network/espnews/schedule/): 403 Client Error: Forbidden for url: https://www.tvinsider.com/network/espnews/schedule/
[WARN] TV guide fetch failed for Golf (https://www.tvinsider.com/network/golf-channel/schedule/): 403 Client Error: Forbidden for url: https://www.tvinsider.com/network/golf-channel/schedule/
[WARN] TV guide fetch failed for FS1 (https://www.tvinsider.com/network/fox-sports-1/sche

In [3]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)